W4M2 — Yellow Taxi가 자율주행차였다면?
==============================================================
유휴 AV를 활용한 검사관 "헛걸음 출동" 방지 서비스

분석 대상
  2024년 3월 · 맨해튼 · descriptor = Pothole · status = Closed
  그중 resolution_description이 '헛걸음' 사유인 신고건

답하려는 질문
  "이 헛걸음 신고들이 접수된 그 시각, 그 존에 유휴 AV가 있었는가?
   있었다면 검사관 출동 몇 건을 사전에 걸러낼 수 있었는가?
   그리고 그걸 하는 데 차가 몇 대나 필요한가?"

헤드라인 지표
  방지 가능 건수 / 방지율
  필요 차량-시간 · 유휴 용량 대비 사용률 · 최소 필요 대수

주의
  resolution_description은 종결 후에야 생기는 사후 정보다.
  배차 입력이 아니라 '사후 가치 평가 라벨'로만 쓴다.
  (유휴 용량이 수요를 압도하므로 전체 신고를 대상으로 해도 결과는 거의 같다)

In [2]:
# ==============================================================
# [C1] 설정 & Spark 세션
# ==============================================================
import os
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"      # macOS bind 오류 회피, import보다 먼저

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

TRIP_PATH    = "data/yellow_2024-03.parquet"
ZONE_PATH    = "data/taxi_zone_lookup.csv"
POTHOLE_ROOT = "warehouse/nyc311"               # year=/month= 파티션 루트
SHAPE_PATH   = "zones/taxi_zones.shp"
OUT_DIR      = "output"

TARGET_BOROUGH = "Manhattan"
YEAR, MONTH = 2024, 3
MONTH_START = f"{YEAR}-{MONTH:02d}-01"
N_DAYS = 31

MIN_DURATION_MIN, MAX_DURATION_MIN = 1.0, 180.0
MAX_DISTANCE_MI = 100.0

INSPECT_MIN = 3.0        # 검증 1건당 소요 시간(분) — 가정값, C13에서 민감도 분석

spark = (SparkSession.builder
         .appName("av-pothole-prevention")
         .master("local[*]")
         .config("spark.driver.bindAddress", "127.0.0.1")
         .config("spark.driver.host", "localhost")
         .config("spark.driver.memory", "6g")
         .config("spark.sql.shuffle.partitions", "64")
         .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
os.makedirs(OUT_DIR, exist_ok=True)
print("Spark", spark.version)

The operation couldn’t be completed. Unable to locate a Java Runtime.
Please visit http://www.java.com for information on installing Java.

/Users/admin/Code/Softeer_DE_wiki/missions/W4/venv/lib/python3.11/site-packages/pyspark/bin/spark-class: line 97: CMD: bad array subscript
head: illegal line count -- -1


RuntimeError: Java gateway process exited before sending its port number

In [ ]:
# ==============================================================
# [C2] 존 룩업 → 맨해튼 존 추출
# ==============================================================
zones = (
    spark.read.option("header", True).csv(ZONE_PATH)
    .select(F.col("LocationID").cast("int").alias("zone_id"),
            F.col("Borough").alias("borough"),
            F.col("Zone").alias("zone_name"))
)

manhattan = zones.filter(F.col("borough") == TARGET_BOROUGH).cache()
manhattan_ids = [r["zone_id"] for r in manhattan.select("zone_id").collect()]
print(f"[C2] {TARGET_BOROUGH} 존 {len(manhattan_ids)}개")

In [ ]:
# ==============================================================
# [C3] TLC 트립 로드 & 정제
# ==============================================================
trips_raw = spark.read.parquet(TRIP_PATH).select(
    F.col("tpep_pickup_datetime").alias("pickup_ts"),
    F.col("tpep_dropoff_datetime").alias("dropoff_ts"),
    F.col("PULocationID").cast("int").alias("pu_zone"),
    F.col("DOLocationID").cast("int").alias("do_zone"),
    F.col("trip_distance").cast("double").alias("distance"),
)

trips = (
    trips_raw
    .filter((F.year("pickup_ts") == YEAR) & (F.month("pickup_ts") == MONTH))
    # TIMESTAMP_NTZ는 long 캐스팅이 막힌다. timestampdiff는 벽시계 기준이라
    # 서머타임 전환(2024-03-10) 영향도 받지 않는다.
    .withColumn("duration_min",
                F.expr("timestampdiff(SECOND, pickup_ts, dropoff_ts)") / 60.0)
    .filter(F.col("duration_min").between(MIN_DURATION_MIN, MAX_DURATION_MIN))
    .filter(F.col("distance").between(0.01, MAX_DISTANCE_MI))
    .filter(F.col("pu_zone").isin(manhattan_ids))
    .filter(F.col("do_zone").isin(manhattan_ids))
    .cache()
)

print(f"[C3] 원본 {trips_raw.count():,} → 정제 후 {trips.count():,}")

In [ ]:
# ==============================================================
# [C4] 시간 슬롯 펼치기 → 점유 차량 수
#   09:50~10:30 트립 → 9시 슬롯에 10분, 10시 슬롯에 30분
#   슬롯 총 주행분 / 60 = 그 시간의 평균 동시 점유 차량 수
# ==============================================================
occupancy = (
    trips
    .withColumn("pu_hr", F.date_trunc("hour", "pickup_ts"))
    .withColumn("do_hr", F.date_trunc("hour", "dropoff_ts"))
    .withColumn("slot", F.explode(F.expr("sequence(pu_hr, do_hr, interval 1 hour)")))
    .withColumn("slot_end", F.expr("slot + interval 1 hour"))
    .withColumn("ov_start", F.greatest("pickup_ts", "slot"))
    .withColumn("ov_end",   F.least("dropoff_ts", "slot_end"))
    .withColumn("busy_min", F.expr("timestampdiff(SECOND, ov_start, ov_end)") / 60.0)
    .filter(F.col("busy_min") > 0)
    .select("slot", F.col("pu_zone").alias("zone_id"), "busy_min")
    .cache()
)

print(f"[C4] 슬롯 확장 후 {occupancy.count():,} 행")

In [ ]:
# ==============================================================
# [C5] 시간대 프로파일 — 유휴율 하한 곡선
#   F ≥ max(점유)  ⟹  유휴율(h) ≥ 1 − 점유(h)/max(점유)
#   플릿 규모 F를 가정하지 않고 하한을 단정할 수 있다.
# ==============================================================
w_daytype = Window.partitionBy("daytype")

hourly = (
    occupancy.groupBy("slot")
    .agg((F.sum("busy_min") / 60.0).alias("occupied"))
    .withColumn("hour", F.hour("slot"))
    .withColumn("daytype", F.when(F.dayofweek("slot").isin([1, 7]), F.lit("weekend"))
                            .otherwise(F.lit("weekday")))
    .groupBy("daytype", "hour")
    .agg(F.avg("occupied").alias("occupied"))
    .withColumn("peak", F.max("occupied").over(w_daytype))
    .withColumn("idle_rate", F.lit(1.0) - F.col("occupied") / F.col("peak"))
    .orderBy("daytype", "hour")
    .cache()
)

hourly.filter(F.col("daytype") == "weekday") \
      .select("hour", F.round("occupied", 1).alias("점유대수"),
              F.round(F.col("idle_rate") * 100, 1).alias("유휴율하한%")) \
      .show(24, truncate=False)

In [ ]:
# ==============================================================
# [C6] 일자 × 시각 × 존 유휴 차량
#   존 피크 = 그 존이 실제로 감당해 본 최대 동시 점유량
#           = 그 존에 배치 가능한 차량 규모의 하한 (F 가정 불필요)
# ==============================================================
occ_zone_slot = (
    occupancy.groupBy("slot", "zone_id")
    .agg((F.sum("busy_min") / 60.0).alias("occupied"))
    .withColumn("date", F.to_date("slot"))
    .withColumn("hour", F.hour("slot"))
    .select("date", "hour", "zone_id", "occupied")
)

# 점유 0인 슬롯은 행 자체가 없다 → 격자를 먼저 만들어야 유휴가 누락되지 않는다
dates = spark.range(N_DAYS).select(
    F.date_add(F.lit(MONTH_START).cast("date"), F.col("id").cast("int")).alias("date")
).filter(F.month("date") == MONTH)
hours = spark.range(24).select(F.col("id").cast("int").alias("hour"))
grid = dates.crossJoin(hours).crossJoin(manhattan.select("zone_id"))

w_zone = Window.partitionBy("zone_id")

occ_daily = (
    grid.join(occ_zone_slot, ["date", "hour", "zone_id"], "left")
        .fillna(0.0, subset=["occupied"])
        .withColumn("zone_peak", F.max("occupied").over(w_zone))
        .withColumn("idle_vehicles",
                    F.greatest(F.col("zone_peak") - F.col("occupied"), F.lit(0.0)))
        .join(manhattan.select("zone_id", "zone_name"), "zone_id")
        .cache()
)

IDLE_POOL = occ_daily.agg(F.sum("idle_vehicles")).first()[0]
print(f"[C6] occ_daily {occ_daily.count():,} 행 · "
      f"가용 유휴 차량-시간 총량 {IDLE_POOL:,.0f} h")

In [ ]:
# ==============================================================
# [C7] 존 내부 이동시간 → 차량 1대의 시간당 처리 건수
#   처리율(z) = 60 / (존내부 이동시간 + 검사시간)
# ==============================================================
intra = (
    trips.filter(F.col("pu_zone") == F.col("do_zone"))
    .groupBy(F.col("pu_zone").alias("zone_id"))
    .agg(F.expr("percentile_approx(duration_min, 0.5)").alias("intra_min"),
         F.count("*").alias("n_intra"))
    .cache()
)

DEFAULT_INTRA = intra.agg(F.expr("percentile_approx(intra_min, 0.5)")).first()[0]
print(f"[C7] 존 내부 이동시간 중앙값 {DEFAULT_INTRA:.1f}분 (결측 존 기본값)")

In [ ]:
# ==============================================================
# [C8] 존별 유휴 재고 — 히트맵·지도용 근거 자료
#   순유입 = 하차 − 승차, 자정부터 누적
# ==============================================================
pickups = (trips.withColumn("date", F.to_date("pickup_ts"))
                .withColumn("hour", F.hour("pickup_ts"))
                .groupBy("date", "hour", F.col("pu_zone").alias("zone_id"))
                .agg(F.count("*").alias("pickups")))

dropoffs = (trips.withColumn("date", F.to_date("dropoff_ts"))
                 .withColumn("hour", F.hour("dropoff_ts"))
                 .groupBy("date", "hour", F.col("do_zone").alias("zone_id"))
                 .agg(F.count("*").alias("dropoffs")))

w_cum_day = (Window.partitionBy("date", "zone_id").orderBy("hour")
                   .rowsBetween(Window.unboundedPreceding, Window.currentRow))

zone_stock = (
    grid.join(pickups,  ["date", "hour", "zone_id"], "left")
        .join(dropoffs, ["date", "hour", "zone_id"], "left")
        .fillna(0, subset=["pickups", "dropoffs"])
        .withColumn("net_inflow", F.col("dropoffs") - F.col("pickups"))
        .withColumn("idle_stock", F.sum("net_inflow").over(w_cum_day))
        .withColumn("daytype", F.when(F.dayofweek("date").isin([1, 7]), F.lit("weekend"))
                                .otherwise(F.lit("weekday")))
        .groupBy("daytype", "zone_id", "hour")
        .agg(F.avg("idle_stock").alias("avg_idle_stock"))
        .join(manhattan.select("zone_id", "zone_name"), "zone_id")
        .cache()
)

print(f"[C8] zone_stock {zone_stock.count():,} 행")

In [ ]:
# ==============================================================
# [C9] 311 파임 신고 — 파티션 프루닝 로드 & 유형 확인
# ==============================================================
rep_raw = (spark.read.parquet(POTHOLE_ROOT)
           .filter((F.col("year") == YEAR) & (F.col("month") == MONTH)))

print(f"[C9] {YEAR}-{MONTH:02d} 파티션 {rep_raw.count():,} 행")

# warehouse에는 311 민원 전체가 들어 있으므로 대상을 좁혀야 한다
rep_raw.filter(F.col("agency") == "DOT") \
       .groupBy("complaint_type", "descriptor").count() \
       .orderBy(F.desc("count")).show(20, truncate=False)

In [ ]:
# ==============================================================
# [C10] 필터링 + 헛걸음 라벨링
#   맨해튼 · Pothole · Closed · 고속도로 제외
# ==============================================================
WASTED_KEYWORDS = [
    "determined that this complaint is a duplicate",               # 중복 신고
    "did not find the reported problem",                           # 문제 미발견
    "found that the problem was fixed",                            # 이미 수리됨
    "in compliance with Department of Transportation standards",   # 기준 부합·비위험
    "not within its jurisdiction",                                 # 소관 아님 (MTA 포함)
    "referred it to the Arterial Division",                        # 타 부서 이관
    "referred it to the Bridge Division",
    "reported condition was not found",                            # 상태 미발견, 무조치
]

# created_date가 string이므로 여러 포맷을 순차 시도
created_expr = F.coalesce(
    F.to_timestamp("created_date", "yyyy-MM-dd'T'HH:mm:ss.SSS"),
    F.to_timestamp("created_date", "yyyy-MM-dd HH:mm:ss"),
    F.to_timestamp("created_date", "MM/dd/yyyy hh:mm:ss a"),
    F.col("created_date").cast("timestamp"),
)

rep = (
    rep_raw
    .filter(F.col("agency") == "DOT")
    .filter(F.lower(F.col("descriptor")).contains("pothole"))
    .filter(F.lower(F.col("status")) == "closed")
    .select(
        F.col("unique_key").cast("string").alias("report_id"),
        created_expr.alias("created_ts"),
        F.col("latitude").cast("double").alias("lat"),
        F.col("longitude").cast("double").alias("lon"),
        F.col("borough"),
        F.col("bridge_highway_name").alias("highway"),
        F.col("resolution_description").alias("resolution"),
    )
    .filter(F.col("created_ts").isNotNull())
    .filter(F.col("lat").isNotNull() & F.col("lon").isNotNull())
    .filter(F.col("highway").isNull())
    .filter(F.upper(F.col("borough")) == TARGET_BOROUGH.upper())
    .withColumn("date", F.to_date("created_ts"))
    .withColumn("hour", F.hour("created_ts"))
    .cache()
)

# NULL 안전: False | NULL = NULL 이 되어 조용히 누락되는 것을 막는다
wasted_expr = F.lit(False)
for kw in WASTED_KEYWORDS:
    wasted_expr = wasted_expr | F.coalesce(
        F.col("resolution").contains(kw), F.lit(False))

rep = rep.withColumn("is_wasted", wasted_expr.cast("int"))

print(f"[C10] 맨해튼 Pothole(Closed) {rep.count():,}건")
rep.select(F.min("date").alias("최소일"), F.max("date").alias("최대일")).show()
rep.groupBy("is_wasted").count().show()
rep.filter(F.col("is_wasted") == 1).groupBy("resolution").count() \
   .orderBy(F.desc("count")).show(10, truncate=70)

In [ ]:
# ==============================================================
# [C11] 좌표 → 택시 존 매핑
#   폴리곤 69개라 브로드캐스트 UDF가 실용적
#   (Sedona 사용 가능하면 ST_Contains로 대체 권장)
# ==============================================================
import geopandas as gpd
from shapely.geometry import Point

_shapes = gpd.read_file(SHAPE_PATH).to_crs(epsg=4326)
_shapes["LocationID"] = _shapes["LocationID"].astype(int)
_shapes = _shapes[_shapes["LocationID"].isin(manhattan_ids)]

bc_shapes = spark.sparkContext.broadcast(
    [(int(r.LocationID), r.geometry) for r in _shapes.itertuples()])


@F.udf(IntegerType())
def to_zone(lon, lat):
    if lon is None or lat is None:
        return None
    p = Point(lon, lat)
    for zid, geom in bc_shapes.value:
        if geom.contains(p):
            return zid
    return None


reports = (rep.withColumn("zone_id", to_zone("lon", "lat"))
              .filter(F.col("zone_id").isNotNull())
              .select("report_id", "date", "hour", "zone_id", "resolution", "is_wasted")
              .cache())

N_ALL = reports.count()
N_WASTED = reports.agg(F.sum("is_wasted")).first()[0]
print(f"[C11] 존 매핑 완료 {N_ALL:,}건 (실패 {rep.count()-N_ALL:,}건 제외)")
print(f"      헛걸음 확정 {N_WASTED:,}건 ({N_WASTED/N_ALL*100:.1f}%)")

In [ ]:
# ==============================================================
# [C12] ★ 헛걸음 방지 가능 여부 판정
#   접수된 그 날짜·시각·존에 유휴 AV가 1대 이상 있었는가
# ==============================================================
wasted_rep = reports.filter(F.col("is_wasted") == 1)

cover = (
    wasted_rep
    .join(occ_daily.select("date", "hour", "zone_id", "zone_name", "idle_vehicles"),
          ["date", "hour", "zone_id"], "left")
    .fillna(0.0, ["idle_vehicles"])
    .withColumn("coverable", (F.col("idle_vehicles") >= 1).cast("int"))
    .cache()
)

PREVENTED = cover.agg(F.sum("coverable")).first()[0]
print(f"[C12] 헛걸음 {N_WASTED:,}건 중 방지 가능 {PREVENTED:,}건 "
      f"({PREVENTED/N_WASTED*100:.1f}%)")

In [ ]:
# ==============================================================
# [C13] ★ 소요 자원 — 이게 진짜 헤드라인
#   "할 수 있나"는 이미 답이 나왔다. 남는 질문은 "얼마나 적은 자원으로 되나".
# ==============================================================
def required_hours(inspect_min):
    return (wasted_rep.groupBy("zone_id").agg(F.count("*").alias("n"))
            .join(intra.select("zone_id", "intra_min"), "zone_id", "left")
            .fillna(DEFAULT_INTRA, ["intra_min"])
            .withColumn("veh_h", F.col("n") * (F.col("intra_min") + inspect_min) / 60.0)
            .agg(F.sum("veh_h")).first()[0])


NEED_H = required_hours(INSPECT_MIN)

print("=" * 62)
print(f"  대상: 맨해튼 Pothole 종결 건 중 헛걸음 사유")
print("-" * 62)
print(f"  헛걸음 확정 신고             {N_WASTED:>10,} 건")
print(f"  AV로 방지 가능               {PREVENTED:>10,} 건  "
      f"({PREVENTED/N_WASTED*100:.1f}%)")
print("-" * 62)
print(f"  필요 차량-시간               {NEED_H:>10,.1f} h")
print(f"  가용 유휴 차량-시간          {IDLE_POOL:>10,.0f} h")
print(f"  → 유휴 용량 사용률           {NEED_H/IDLE_POOL*100:>10.4f} %")
print(f"  → 24시간 상시 필요 대수      {NEED_H/(N_DAYS*24):>10.2f} 대")
print(f"  → 심야 6시간(0~6시)만 쓸 때  {NEED_H/(N_DAYS*6):>10.2f} 대")
print("=" * 62)

# 검사시간 가정 민감도
sens_rows = []
for m in [3.0, 5.0, 10.0, 20.0]:
    h = required_hours(m)
    sens_rows.append((m, round(h, 1), round(h / IDLE_POOL * 100, 4),
                      round(h / (N_DAYS * 24), 2)))
sens = spark.createDataFrame(
    sens_rows, ["검사시간_분", "필요차량시간", "유휴사용률_pct", "상시필요대수"])
sens.show(truncate=False)

In [ ]:
# ==============================================================
# [C14] 시각화 준비 — 여기서만 toPandas
# ==============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.colors import TwoSlopeNorm

FONT = None
for cand in ["AppleGothic", "Malgun Gothic", "NanumGothic"]:
    if any(f.name == cand for f in fm.fontManager.ttflist):
        FONT = cand
        plt.rcParams["font.family"] = cand
        break
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 120
BOLD = "normal" if FONT == "AppleGothic" else "bold"   # AppleGothic엔 bold가 없다

cover_pd = cover.toPandas()
cover_pd["date"] = pd.to_datetime(cover_pd["date"])
cover_pd["day"] = cover_pd["date"].dt.day

all_pd    = reports.groupBy("date", "hour").count().toPandas()
hourly_pd = hourly.toPandas()
stock_pd  = zone_stock.toPandas()
sens_pd   = sens.toPandas()

print({k: v.shape for k, v in
       dict(cover=cover_pd, hourly=hourly_pd, stock=stock_pd).items()})

In [ ]:
# ==============================================================
# [C15] ★ 메인 대시보드 — 헛걸음 방지
# ==============================================================
def short_reason(t):
    for k, v in [("duplicate", "중복 신고"), ("did not find", "문제 미발견"),
                 ("problem was fixed", "이미 수리됨"), ("compliance", "기준 부합"),
                 ("jurisdiction", "소관 아님"), ("Arterial", "간선과 이관"),
                 ("Bridge Division", "교량과 이관"),
                 ("condition was not found", "상태 미발견")]:
        if k in str(t):
            return v
    return "기타"


cover_pd["reason"] = cover_pd["resolution"].map(short_reason)

fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 3, height_ratios=[0.8, 1, 1], hspace=0.55, wspace=0.32)

# ── 상단 KPI ──
ax = fig.add_subplot(gs[0, :]); ax.axis("off")
kpis = [("헛걸음 신고", f"{N_WASTED:,}건"),
        ("AV 방지 가능", f"{PREVENTED:,}건"),
        ("방지율", f"{PREVENTED/N_WASTED*100:.1f}%"),
        ("상시 필요 대수", f"{NEED_H/(N_DAYS*24):.2f}대"),
        ("유휴 용량 사용", f"{NEED_H/IDLE_POOL*100:.3f}%")]
for i, (label, val) in enumerate(kpis):
    x = 0.1 + i * 0.2
    c = "#2f855a" if i in (1, 2) else "#2b6cb0"
    ax.text(x, 0.60, val, ha="center", fontsize=24, weight=BOLD, color=c)
    ax.text(x, 0.28, label, ha="center", fontsize=10, color="#666")
ax.text(0.5, 0.0,
        f"2024년 3월 · 맨해튼 · Pothole · 종결 건 중 헛걸음 사유 · "
        f"검사시간 {INSPECT_MIN:.0f}분 가정",
        ha="center", fontsize=8.5, color="#999")

# ── 중단 좌: 접수 시각 분포 vs 유휴율 (핵심 그림) ──
ax = fig.add_subplot(gs[1, :2])
byh = cover_pd.groupby("hour").size().reindex(range(24), fill_value=0)
ax.bar(byh.index, byh.values, color="#c53030", width=0.8, alpha=0.75)
ax.set_xlabel("시각"); ax.set_ylabel("헛걸음 신고 접수 (건)", color="#c53030")
ax.set_xticks(range(0, 24, 2)); ax.grid(axis="y", alpha=0.2)

ax2 = ax.twinx()
s = hourly_pd[hourly_pd["daytype"] == "weekday"].sort_values("hour")
ax2.plot(s["hour"], s["idle_rate"] * 100, lw=2.4, ls="--", color="#2b6cb0")
ax2.set_ylabel("유휴 AV 비율 (%)", color="#2b6cb0"); ax2.set_ylim(0, 100)
ax.set_title("신고는 낮에 들어오고, 유휴 AV는 밤에 남는다 — "
             "사람 검사관으로는 메울 수 없는 시간대", fontsize=11, weight=BOLD)

# ── 중단 우: 헛걸음 사유 구성 ──
ax = fig.add_subplot(gs[1, 2])
rc = cover_pd["reason"].value_counts()
ax.barh(range(len(rc)), rc.values, color="#c53030")
ax.set_yticks(range(len(rc))); ax.set_yticklabels(rc.index, fontsize=8)
ax.invert_yaxis(); ax.set_xlabel("건수")
ax.set_title("무엇을 걸러내는가", fontsize=10, weight=BOLD)
ax.grid(axis="x", alpha=0.2)

# ── 하단 좌: 일자별 발생 / 방지 ──
ax = fig.add_subplot(gs[2, :2])
byd = (cover_pd.groupby("day")
       .agg(total=("coverable", "size"), ok=("coverable", "sum"))
       .reindex(range(1, N_DAYS + 1), fill_value=0))
ax.bar(byd.index, byd["total"], color="#e2e8f0", width=0.8, label="헛걸음 발생")
ax.bar(byd.index, byd["ok"], color="#2f855a", width=0.8, label="AV 방지 가능")
ax.set_xlabel("3월 (일)"); ax.set_ylabel("건수")
ax.set_title("일자별 헛걸음 발생과 방지 가능 건수", fontsize=11, weight=BOLD)
ax.legend(frameon=False, fontsize=9); ax.grid(axis="y", alpha=0.2)

# ── 하단 우: 헛걸음 다발 존 ──
ax = fig.add_subplot(gs[2, 2])
bz = cover_pd.groupby("zone_name").size().nlargest(8)
ax.barh(range(len(bz)), bz.values, color="#2b6cb0")
ax.set_yticks(range(len(bz)))
ax.set_yticklabels([z[:20] for z in bz.index], fontsize=7.5)
ax.invert_yaxis(); ax.set_xlabel("헛걸음 건수")
ax.set_title("헛걸음 다발 존 상위 8", fontsize=10, weight=BOLD)
ax.grid(axis="x", alpha=0.2)

fig.suptitle("유휴 AV를 활용한 검사관 헛걸음 방지 — 맨해튼, 2024년 3월",
             fontsize=15, weight=BOLD, y=0.97)
plt.show()

In [ ]:
# ==============================================================
# [C16] 부록 A — 가정 민감도 (검사시간에 따른 소요 자원)
# ==============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

ax = axes[0]
ax.bar(sens_pd["검사시간_분"].astype(int).astype(str),
       sens_pd["상시필요대수"], color="#2b6cb0", width=0.55)
for i, v in enumerate(sens_pd["상시필요대수"]):
    ax.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=9)
ax.set_xlabel("검증 1건당 소요 시간 (분)"); ax.set_ylabel("24시간 상시 필요 대수")
ax.set_title("가정이 바뀌어도 필요 대수는 미미하다", fontsize=11, weight=BOLD)
ax.grid(axis="y", alpha=0.2)

ax = axes[1]
ax.bar(sens_pd["검사시간_분"].astype(int).astype(str),
       sens_pd["유휴사용률_pct"], color="#2f855a", width=0.55)
for i, v in enumerate(sens_pd["유휴사용률_pct"]):
    ax.text(i, v, f"{v:.3f}%", ha="center", va="bottom", fontsize=9)
ax.set_xlabel("검증 1건당 소요 시간 (분)"); ax.set_ylabel("유휴 용량 사용률 (%)")
ax.set_title("전체 유휴 용량 대비 사용 비중", fontsize=11, weight=BOLD)
ax.grid(axis="y", alpha=0.2)

fig.tight_layout(); plt.show()

In [ ]:
# ==============================================================
# [C17] 부록 B — 존 × 시간대 유휴 재고 히트맵
# ==============================================================
ds = stock_pd[stock_pd["daytype"] == "weekday"]
pivot = ds.pivot_table(index="zone_name", columns="hour",
                       values="avg_idle_stock", aggfunc="mean")
pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

vmax = np.nanpercentile(np.abs(pivot.values), 98)
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

fig, ax = plt.subplots(figsize=(11, max(8, len(pivot) * 0.22)))
im = ax.imshow(pivot.values, aspect="auto", cmap="RdBu_r", norm=norm)
ax.set_xticks(range(0, 24, 2)); ax.set_xticklabels(range(0, 24, 2))
ax.set_yticks(range(len(pivot))); ax.set_yticklabels(pivot.index, fontsize=7)
ax.set_xlabel("시각")
ax.set_title("존 × 시간대 유휴 재고 (평일)\n"
             "붉을수록 빈 차가 쌓임 / 푸를수록 차가 모자람", fontsize=11)
fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02).set_label("누적 순유입 (하차 − 승차)")
fig.tight_layout(); plt.show()

In [ ]:
# ==============================================================
# [C18] 부록 C — 맨해튼 지도 (헛걸음 분포 vs 유휴 분포)
# ==============================================================
gdf = _shapes.rename(columns={"LocationID": "zone_id"})[["zone_id", "geometry"]]

fig, axes = plt.subplots(1, 2, figsize=(13, 9))

# 좌: 헛걸음 신고 밀도
wz = cover_pd.groupby("zone_id").size().rename("wasted").reset_index()
gdf.merge(wz, on="zone_id", how="left").plot(
    column="wasted", cmap="Reds", linewidth=0.3, edgecolor="white",
    ax=axes[0], legend=True, missing_kwds={"color": "#f0f0f0"})
axes[0].set_title("헛걸음 신고 분포", fontsize=12, weight=BOLD)
axes[0].set_axis_off()

# 우: 새벽 3시 유휴 재고
snap = ds[ds["hour"] == 3][["zone_id", "avg_idle_stock"]]
vm = np.nanpercentile(np.abs(snap["avg_idle_stock"]), 98)
gdf.merge(snap, on="zone_id", how="left").plot(
    column="avg_idle_stock", cmap="RdBu_r",
    norm=TwoSlopeNorm(vmin=-vm, vcenter=0, vmax=vm),
    linewidth=0.3, edgecolor="white", ax=axes[1], legend=True,
    missing_kwds={"color": "#f0f0f0"})
axes[1].set_title("새벽 3시 유휴 AV 분포", fontsize=12, weight=BOLD)
axes[1].set_axis_off()

fig.suptitle("헛걸음이 많은 곳과 차가 남는 곳은 겹치는가 (맨해튼, 2024-03)",
             fontsize=14, weight=BOLD, y=0.94)
plt.show()

In [ ]:
# ==============================================================
# [C19] 저장 & 세션 종료
# ==============================================================
cover.select("report_id", "date", "hour", "zone_id", "zone_name",
             "idle_vehicles", "coverable", "resolution") \
     .coalesce(1).write.mode("overwrite").option("header", True) \
     .csv(f"{OUT_DIR}/wasted_coverage")

cover_pd.to_csv(f"{OUT_DIR}/wasted_coverage.csv", index=False)
print(f"[C19] 저장 완료 → {OUT_DIR}/")
spark.stop()